# **Validasi Struktur dan Revisi Variasi Bahasa Lokal**

---

**Project:** KP Agentic AI + LLM  
**Tahap:** Step 12  
**Dataset:** MASSIVE-Agents `id-ID` development

Revisi dibuat berdasarkan tinjauan awal 500 variasi. Fokus perbaikan ada pada code-switching Indonesia–Inggris dan Indonesia–Jawa yang terlalu sederhana. Data asli dan hasil v2 tetap disimpan tanpa perubahan.

## **1. Dataset dan Konfigurasi**

---

Input terdiri dari 397 instruksi Indonesia development dan hasil kandidat variasi v2. Eksperimen hanya memakai development, dengan seed yang tetap.

In [1]:
from __future__ import annotations

import csv
import hashlib
import json
import os
import random
import re
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

SEED = 20261110
TARGET_AUDIT = 100
ROOT_OVERRIDE = None
KATEGORI = ["slang", "singkatan", "typo", "code_switch_id_en", "code_switch_id_jawa"]


def cari_root():
    candidates = []
    if ROOT_OVERRIDE is not None:
        candidates.append(Path(ROOT_OVERRIDE))
    if os.environ.get("KP_AGENTIC_AI_ROOT"):
        candidates.append(Path(os.environ["KP_AGENTIC_AI_ROOT"]))
    current = Path.cwd().resolve()
    candidates.extend([current, *current.parents, Path(r"D:\KP-Agentic-AI")])
    for root in candidates:
        dev = root / "data" / "processed" / "massive_agents_reviewed" / "id-ID_development.jsonl"
        old = root / "results" / "variasi_bahasa_terkontrol" / "development_v2" / "variasi_bahasa_kandidat_v2.csv"
        formal = root / "results" / "variasi_bahasa_terkontrol" / "development_v2" / "kandidat_induk_formal_v2.csv"
        if dev.is_file() and old.is_file() and formal.is_file():
            return root
    raise FileNotFoundError("Dataset development dan hasil development_v2 belum ditemukan pada folder project.")

ROOT = cari_root()
INPUT = ROOT / "data" / "processed" / "massive_agents_reviewed" / "id-ID_development.jsonl"
INPUT_V2 = ROOT / "results" / "variasi_bahasa_terkontrol" / "development_v2" / "variasi_bahasa_kandidat_v2.csv"
INPUT_FORMAL_V2 = ROOT / "results" / "variasi_bahasa_terkontrol" / "development_v2" / "kandidat_induk_formal_v2.csv"
OUTPUT = ROOT / "results" / "variasi_bahasa_terkontrol" / "development_v3"
OUTPUT.mkdir(parents=True, exist_ok=True)

print("Dataset:", INPUT)
print("Kandidat v2:", INPUT_V2)
print("Output v3:", OUTPUT)

Dataset: D:\KP-Agentic-AI\data\processed\massive_agents_reviewed\id-ID_development.jsonl
Kandidat v2: D:\KP-Agentic-AI\results\variasi_bahasa_terkontrol\development_v2\variasi_bahasa_kandidat_v2.csv
Output v3: D:\KP-Agentic-AI\results\variasi_bahasa_terkontrol\development_v3


## **2. Pemeriksaan Sumber Data**

---

Instruksi induk pada v2 dicocokkan kembali dengan dataset asli. Setiap `row_key` dan `group_id` harus menunjuk ke record development yang sama.

In [2]:
with INPUT.open("r", encoding="utf-8") as handle:
    records = [json.loads(line) for line in handle if line.strip()]

with INPUT_V2.open("r", encoding="utf-8-sig", newline="") as handle:
    old_variants = list(csv.DictReader(handle))
with INPUT_FORMAL_V2.open("r", encoding="utf-8-sig", newline="") as handle:
    formal_candidates = list(csv.DictReader(handle))

assert len(records) == 397, "Jumlah record development berubah."
assert len(old_variants) == 931, "Jumlah kandidat v2 berbeda dari hasil audit sebelumnya."
assert len(formal_candidates) == 359, "Daftar induk formal v2 berbeda dari hasil audit sebelumnya."

source = {row["row_key"]: row for row in records}
assert len(source) == len(records)
parents = {}
for item in formal_candidates:
    original = source[item["row_key_induk"]]
    assert item["instruksi_id_asli"] == original["question"]
    assert item["group_id"] == original["group_id"]
    assert item["status_formal"] == "BELUM_VALIDASI_MANUAL"
    parents[original["row_key"]] = original

for item in old_variants:
    original = parents[item["row_key_induk"]]
    assert item["instruksi_id_asli"] == original["question"]
    assert item["group_id"] == original["group_id"]
    assert item["split"] == original["split"] == "development"
    assert json.loads(item["gt_json"]) == original["gt"]

old_by_id = {r["variant_id"]: r for r in old_variants}
assert len(old_by_id) == len(old_variants)
print("Record sumber:", len(records))
print("Induk variasi v2:", len(parents))
print("Kandidat v2:", len(old_variants))

Record sumber: 397
Induk variasi v2: 359
Kandidat v2: 931


## **3. Perbaikan Aturan Code-Switching**

---

Aturan `please` dan `tulung` sebagai tambahan di awal instruksi tidak digunakan lagi. Pergantian dilakukan pada frasa yang menyatakan tindakan atau pertanyaan, tanpa mengganti nama orang, lokasi, waktu, atau nilai argumen GT.

Perubahan kode tetap dinilai sebagai **kandidat**, bukan otomatis dianggap alami atau setara secara makna.

In [3]:
# Aturan sengaja dibatasi ke frasa yang cukup jelas.
# Pola yang berpotensi memerlukan penafsiran tambahan tidak dipaksakan.
RULES = {
    "code_switch_id_en": [
        (r"^putar playlist\b", "play playlist", "frasa_tindakan", "sedang"),
        (r"^putar lagu\b", "play lagu", "frasa_tindakan", "sedang"),
        (r"^mainkan lagu\b", "play lagu", "frasa_tindakan", "sedang"),
        (r"^mainkan musik\b", "play musik", "frasa_tindakan", "sedang"),
        (r"^matikan lampu\b", "turn off lampu", "frasa_tindakan", "perlu_review_bahasa"),
        (r"^nyalakan lampu\b", "turn on lampu", "frasa_tindakan", "perlu_review_bahasa"),
        (r"^atur alarm\b", "set alarm", "frasa_tindakan", "sedang"),
        (r"^hapus alarm\b", "delete alarm", "frasa_tindakan", "sedang"),
        (r"^cek email\b", "check email", "frasa_tindakan", "sedang"),
        (r"^periksa email\b", "check email", "frasa_tindakan", "sedang"),
        (r"^turunkan volume\b", "lower volume", "frasa_tindakan", "sedang"),
        (r"^naikkan volume\b", "increase volume", "frasa_tindakan", "sedang"),
        (r"^putar musik\b", "play musik", "frasa_tindakan", "sedang"),
        (r"^matikan alarm\b", "turn off alarm", "frasa_tindakan", "perlu_review_bahasa"),
        (r"^atur volume\b", "set volume", "frasa_tindakan", "sedang"),
    ],
    "code_switch_id_jawa": [
        (r"^tolong matikan lampu\b", "tulung pateni lampu", "frasa_ngoko", "perlu_penutur"),
        (r"^tolong nyalakan lampu\b", "tulung uripna lampu", "frasa_ngoko", "perlu_penutur"),
        (r"^tolong hapus alarm\b", "tulung busak alarm", "frasa_ngoko", "perlu_penutur"),
        (r"^matikan lampu\b", "pateni lampu", "frasa_ngoko", "perlu_penutur"),
        (r"^nyalakan lampu\b", "uripna lampu", "frasa_ngoko", "perlu_penutur"),
        (r"^hapus alarm\b", "busak alarm", "frasa_ngoko", "perlu_penutur"),
        (r"^putar lagu\b", "puterna lagu", "frasa_ngoko", "perlu_penutur"),
        (r"^mainkan lagu\b", "puterna lagu", "frasa_ngoko", "perlu_penutur"),
        (r"^jam berapa\b", "jam piro", "frasa_pertanyaan", "perlu_penutur"),
        (r"^berapa harga\b", "piro regane", "frasa_pertanyaan", "perlu_penutur"),
        (r"^berapa lama\b", "pira suwene", "frasa_pertanyaan", "perlu_penutur"),
        (r"^bagaimana cara\b", "piye carane", "frasa_pertanyaan", "perlu_penutur"),
        (r"\bdi mana\b", "neng endi", "frasa_pertanyaan", "perlu_penutur"),
        (r"\bsekarang\b", "saiki", "keterangan_waktu", "perlu_penutur"),
        (r"\bbesok\b", "sesuk", "keterangan_waktu", "perlu_penutur"),
    ],
}


def gt_text_values(obj):
    if isinstance(obj, dict):
        for value in obj.values():
            yield from gt_text_values(value)
    elif isinstance(obj, (list, tuple)):
        for value in obj:
            yield from gt_text_values(value)
    elif isinstance(obj, str) and obj.strip():
        yield obj.strip()


def gt_spans(row):
    text = row["question"]
    spans = []
    for value in set(gt_text_values(row["gt"])):
        if value.lower() in {"current", "now", "null", "none"}:
            continue
        rgx = re.compile(r"(?<!\w)" + re.escape(value) + r"(?!\w)", re.I)
        spans.extend((m.start(), m.end()) for m in rgx.finditer(text))
    return spans


def replace_controlled(row, category):
    text = row["question"]
    protected = gt_spans(row)
    for pattern, replacement, subtype, review_flag in RULES[category]:
        match = re.search(pattern, text, re.I)
        if match is None:
            continue
        if any(match.start() < right and match.end() > left for left, right in protected):
            continue
        new = text[:match.start()] + replacement + text[match.end():]
        if new != text:
            return new, text[match.start():match.end()], replacement, subtype, review_flag
    return None

print("Jumlah aturan EN:", len(RULES["code_switch_id_en"]))
print("Jumlah aturan Jawa:", len(RULES["code_switch_id_jawa"]))

Jumlah aturan EN: 15
Jumlah aturan Jawa: 15


## **4. Pembentukan Kandidat Revisi**

---

Kategori slang, singkatan, dan typo mengikuti kandidat v2. Dua kategori code-switching dibuat ulang dari instruksi asli. Tidak ada penggantian dengan tambahan satu kata di awal.

In [4]:
revised = []
changes = []
missing = []

for row_key, parent in parents.items():
    for category in KATEGORI:
        variant_id = row_key + "::" + category
        old = old_by_id.get(variant_id)
        if category in {"slang", "singkatan", "typo"}:
            if old is not None:
                item = dict(old)
                item["versi_generator"] = "v2_tetap_belum_validasi"
                item["flag_review"] = "perlu_validasi_manusia"
                revised.append(item)
            continue

        candidate = replace_controlled(parent, category)
        if candidate is None:
            missing.append({
                "row_key_induk": row_key, "kategori": category,
                "alasan": "tidak ada pola frasa yang sesuai atau frasa dilindungi GT"
            })
            if old is not None:
                changes.append({
                    "variant_id": variant_id, "kategori": category,
                    "instruksi_lama": old["instruksi_variasi"], "instruksi_baru": "",
                    "status": "DIHAPUS_DARI_KANDIDAT", "alasan": "aturan terlalu sederhana / tidak ada aturan baru"
                })
            continue

        new_text, old_span, new_span, subtype, review_flag = candidate
        item = {
            "variant_id": variant_id,
            "row_key_induk": parent["row_key"],
            "source_record_id": parent["source_record_id"],
            "group_id": parent["group_id"],
            "split": parent["split"],
            "language_code": parent["language_code"],
            "kategori": category,
            "subtipe": subtype,
            "tingkat_review": review_flag,
            "instruksi_id_asli": parent["question"],
            "instruksi_variasi": new_text,
            "aturan_asal": old_span,
            "aturan_pengganti": new_span,
            "fungsi_gt": next(iter(parent["gt"])),
            "gt_json": json.dumps(parent["gt"], sort_keys=True, ensure_ascii=False),
            "status_validasi": "BELUM_VALIDASI_MANUSIA",
            "versi_generator": "v3_frasa_terkontrol",
            "flag_review": review_flag,
        }
        revised.append(item)
        changes.append({
            "variant_id": variant_id,
            "kategori": category,
            "instruksi_lama": old["instruksi_variasi"] if old else "",
            "instruksi_baru": new_text,
            "status": "DIGANTI" if old else "BARU_DIBUAT",
            "alasan": "penggantian frasa dengan aturan terkontrol"
        })

assert len(revised) == len({item["variant_id"] for item in revised})
print("Kandidat v2:", len(old_variants))
print("Kandidat v3:", len(revised))
for cat in KATEGORI:
    print(f"{cat:<22}", sum(r["kategori"] == cat for r in revised))

Kandidat v2: 931
Kandidat v3: 671
slang                  139
singkatan              187
typo                   304
code_switch_id_en      20
code_switch_id_jawa    21


## **5. Pemeriksaan GT, Group ID, dan Split**

---

Setiap kandidat dibandingkan kembali dengan record asli. Pemeriksaan ini memastikan struktur data tidak berubah, tetapi belum menilai kewajaran variasi dan kesetaraan makna.

In [5]:
checks = []
for item in revised:
    parent = source[item["row_key_induk"]]
    checks.append({
        "variant_id": item["variant_id"],
        "gt_sama": json.loads(item["gt_json"]) == parent["gt"],
        "fungsi_sama": item["fungsi_gt"] == next(iter(parent["gt"])),
        "group_id_sama": item["group_id"] == parent["group_id"],
        "split_development": item["split"] == parent["split"] == "development",
        "instruksi_induk_sama": item["instruksi_id_asli"] == parent["question"],
        "variasi_berbeda": item["instruksi_variasi"] != parent["question"],
    })

assert all(all(v for k, v in c.items() if k != "variant_id") for c in checks)
assert len({r["group_id"] for r in parents.values()}) == len(parents)

print("Pemeriksaan struktur:", "PASS")
print("Validasi semantik:", "BELUM")

Pemeriksaan struktur: PASS
Validasi semantik: BELUM


## **6. Sampel Audit dan Keterbatasan**

---

Sampel diambil maksimal 100 per kategori dengan satu `group_id` per kategori. Jika kandidat tidak mencapai 100, kekurangannya dilaporkan secara terbuka. Data yang belum diperiksa manusia tidak digunakan sebagai hasil final eksperimen ketahanan model.

In [6]:
rng = random.Random(SEED)
samples = []
summary = []
for cat in KATEGORI:
    category_rows = [dict(r) for r in revised if r["kategori"] == cat]
    rng.shuffle(category_rows)
    seen = set()
    selected = []
    for r in category_rows:
        if r["group_id"] in seen:
            continue
        seen.add(r["group_id"])
        selected.append(r)
        if len(selected) == TARGET_AUDIT:
            break
    for item in selected:
        samples.append({
            **item,
            "makna_tetap": "BELUM_DIREVIEW",
            "entitas_waktu_tetap": "BELUM_DIREVIEW",
            "bahasa_alami": "BELUM_DIREVIEW",
            "fungsi_gt_masih_sesuai": "BELUM_DIREVIEW",
            "keputusan_manusia": "BELUM_DIREVIEW",
            "alasan_manual": "",
        })
    summary.append({
        "kategori": cat,
        "kandidat_v2": sum(x["kategori"] == cat for x in old_variants),
        "kandidat_v3": len(category_rows),
        "sampel_audit": len(selected),
        "target": TARGET_AUDIT,
        "kekurangan": max(0, TARGET_AUDIT - len(selected)),
    })

for item in summary:
    print(f'{item["kategori"]:<22} kandidat={item["kandidat_v3"]:<4} sampel={item["sampel_audit"]:<4} kurang={item["kekurangan"]}')
print("Jumlah sampel:", len(samples))

slang                  kandidat=139  sampel=100  kurang=0
singkatan              kandidat=187  sampel=100  kurang=0
typo                   kandidat=304  sampel=100  kurang=0
code_switch_id_en      kandidat=20   sampel=20   kurang=80
code_switch_id_jawa    kandidat=21   sampel=21   kurang=79
Jumlah sampel: 341


## **7. Penyimpanan Hasil**

---

Output v3 disimpan pada folder terpisah. File v2, dataset sumber, dan hasil prediksi baseline tidak ditimpa.

In [7]:
def save_csv(rows, name):
    if not rows:
        return None
    path = OUTPUT / name
    with path.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    return path

outputs = [
    save_csv(revised, "kandidat_variasi_development_v3.csv"),
    save_csv(changes, "perubahan_dari_v2.csv"),
    save_csv(missing, "kategori_tidak_terbentuk_v3.csv"),
    save_csv(checks, "pemeriksaan_konsistensi_v3.csv"),
    save_csv(samples, "sampel_validasi_v3.csv"),
    save_csv(summary, "ringkasan_v2_vs_v3.csv"),
]

metadata = {
    "run": "step12_revisi_variasi_bahasa_v3",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "input_sha256": hashlib.sha256(INPUT.read_bytes()).hexdigest(),
    "v2_sha256": hashlib.sha256(INPUT_V2.read_bytes()).hexdigest(),
    "input_rows": len(records),
    "parent_rows": len(parents),
    "variants_v2": len(old_variants),
    "variants_v3": len(revised),
    "selected_audit": len(samples),
    "counts": {x["kategori"]: x["kandidat_v3"] for x in summary},
    "audit_shortfall": {x["kategori"]: x["kekurangan"] for x in summary},
    "gt_modified": False,
    "test_used": False,
    "human_validation": "pending",
    "approved_for_model_evaluation": False,
    "generator_model_used": False,
    "note": "Pemeriksaan ini hanya memastikan integritas struktur. Kealamian dan makna memerlukan validasi manusia.",
}
meta_path = OUTPUT / "metadata_development_v3.json"
meta_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
outputs.append(meta_path)

for path in outputs:
    if path:
        print(path.name)
print("Selesai menyimpan hasil pada:", OUTPUT)

kandidat_variasi_development_v3.csv
perubahan_dari_v2.csv
kategori_tidak_terbentuk_v3.csv
pemeriksaan_konsistensi_v3.csv
sampel_validasi_v3.csv
ringkasan_v2_vs_v3.csv
metadata_development_v3.json
Selesai menyimpan hasil pada: D:\KP-Agentic-AI\results\variasi_bahasa_terkontrol\development_v3


## **8. Kesimpulan Sementara**

---

Revisi mengurangi variasi yang hanya menambahkan penanda kesopanan dan mempertahankan informasi GT serta split. Jumlah kandidat dapat lebih sedikit karena aturan diperketat. Tahap validasi bahasa dan makna tetap diperlukan sebelum hasil digunakan untuk menguji penurunan akurasi model.